In [1]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("../data/cleaned_transcripts.csv")

# Load queries
queries_df = pd.read_csv("../data/search_queries.csv")

# Load mapping (ground truth)
mapping_df = pd.read_csv("../data/query_video_mapping.csv")

print(df.shape)
print(queries_df.shape)
print(mapping_df.shape)

(295, 4)
(69, 1)
(69, 2)


Load models

In [2]:
from sentence_transformers import SentenceTransformer

models = {
    "MiniLM": SentenceTransformer("all-MiniLM-L6-v2"),
    "MPNet": SentenceTransformer("all-mpnet-base-v2"),
    "MultiQA": SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")
}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step 3: Generate embeddings

In [3]:
texts = df["transcript"].fillna("").tolist()

embeddings = {}

for name, model in models.items():
    print(f"Generating embeddings for {name}...")
    embeddings[name] = model.encode(texts, show_progress_bar=True)

Generating embeddings for MiniLM...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Generating embeddings for MPNet...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Generating embeddings for MultiQA...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Step 4: Query embeddings

In [4]:
query_texts = queries_df["query"].tolist()

query_embeddings = {}

for name, model in models.items():
    query_embeddings[name] = model.encode(query_texts)

Step 5: Similarity methods

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def get_cosine_scores(q_emb, doc_emb):
    return cosine_similarity(q_emb, doc_emb)

Step 6: Distance methods

In [6]:
from scipy.spatial.distance import cdist

def get_distance_scores(q_emb, doc_emb, metric):
    return cdist(q_emb, doc_emb, metric=metric)

 Step 7: Ranking (Top-K)

In [7]:
def get_top_k(scores, k=5, reverse=True):
    results = []
    
    for i, row in enumerate(scores):
        if reverse:
            top_idx = row.argsort()[-k:][::-1]  # similarity
        else:
            top_idx = row.argsort()[:k]         # distance
        
        results.append(top_idx)
    
    return results

Step 8: Evaluation

In [8]:
def evaluate(top_k_indices, mapping_df):
    top1 = 0
    top3 = 0
    top5 = 0
    
    for i, row in mapping_df.iterrows():
        true_vid = row["relevant_video_id"]
        predicted = top_k_indices[i]
        
        predicted_vids = df.iloc[predicted]["video_id"].values
        
        if true_vid == predicted_vids[0]:
            top1 += 1
        
        if true_vid in predicted_vids[:3]:
            top3 += 1
        
        if true_vid in predicted_vids[:5]:
            top5 += 1
    
    total = len(mapping_df)
    
    return {
        "Top-1": top1 / total,
        "Top-3": top3 / total,
        "Top-5": top5 / total
    }

Step 9: Run evaluation

In [9]:
results = []

for name in models.keys():
    
    print(f"\nEvaluating {name} with Cosine...")
    
    scores = get_cosine_scores(query_embeddings[name], embeddings[name])
    
    top_k = get_top_k(scores, k=5, reverse=True)
    
    metrics = evaluate(top_k, mapping_df)
    
    results.append({
        "Model": name,
        "Metric": "Cosine",
        **metrics
    })


Evaluating MiniLM with Cosine...

Evaluating MPNet with Cosine...

Evaluating MultiQA with Cosine...


Step 10: Compare results

In [10]:
results_df = pd.DataFrame(results)
print(results_df)

     Model  Metric     Top-1     Top-3     Top-5
0   MiniLM  Cosine  0.000000  0.028986  0.028986
1    MPNet  Cosine  0.014493  0.028986  0.028986
2  MultiQA  Cosine  0.000000  0.014493  0.014493


In [11]:
texts = (df["title"] + " " + df["transcript"].str[:1000]).fillna("").tolist()

In [12]:
embeddings = {}

for name, model in models.items():
    print(f"Generating embeddings for {name}...")
    embeddings[name] = model.encode(texts, show_progress_bar=True)

Generating embeddings for MiniLM...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Generating embeddings for MPNet...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Generating embeddings for MultiQA...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]